In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = "/content/delhi_emergency_demand_dataset.csv"
df = pd.read_csv(file_path)

In [ ]:
df.head()

,date,hour,zone_id,call_count,PM2.5,PM10,AQI,temperature,humidity,population_density,elderly_pct
0,2015-12-10,15,Z22,1,211.95,386.66,383.0,19.0,77.25,1.865327e+06,0.08
1,2015-12-10,16,Z04,1,211.95,386.66,383.0,19.0,77.25,1.865327e+06,0.08
2,2015-12-10,16,Z06,1,211.95,386.66,383.0,19.0,77.25,1.865327e+06,0.08
3,2015-12-10,16,Z11,1,211.95,386.66,383.0,19.0,77.25,1.865327e+06,0.08
4,2015-12-10,16,Z20,1,211.95,386.66,383.0,19.0,77.25,1.865327e+06,0.08


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256931 entries, 0 to 256930
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   date                256931 non-null  object 
 1   hour                256931 non-null  int64  
 2   zone_id             256931 non-null  object 
 3   call_count          256931 non-null  int64  
 4   PM2.5               256931 non-null  float64
 5   PM10                256931 non-null  float64
 6   AQI                 256931 non-null  float64
 7   temperature         256931 non-null  float64
 8   humidity            256931 non-null  float64
 9   population_density  256931 non-null  float64
 10  elderly_pct         256931 non-null  float64
dtypes: float64(7), int64(2), object(2)
memory usage: 21.6+ MB


In [ ]:
df.describe()

,hour,call_count,PM2.5,PM10,AQI,temperature,humidity,population_density,elderly_pct
count,256931.000000,256931.000000,256931.000000,256931.000000,256931.000000,256931.000000,256931.000000,2.569310e+05,2.569310e+05
mean,12.583413,1.294075,116.951490,234.626192,249.142252,13.651304,90.958703,1.865327e+06,8.000000e-02
std,6.143553,0.606371,85.908302,122.311990,125.167374,7.642305,18.529512,5.704129e-06,3.008294e-13
min,0.000000,1.000000,10.240000,18.590000,29.000000,10.000000,18.466667,1.865327e+06,8.000000e-02
25%,8.000000,1.000000,54.010000,134.460000,138.000000,10.000000,100.000000,1.865327e+06,8.000000e-02
50%,13.000000,1.000000,92.460000,223.560000,241.000000,10.000000,100.000000,1.865327e+06,8.000000e-02
75%,18.000000,1.000000,152.990000,309.850000,338.500000,10.000000,100.000000,1.865327e+06,8.000000e-02
max,23.000000,16.000000,685.360000,796.880000,716.000000,38.272727,100.000000,1.865327e+06,8.000000e-02


Add Variation to elderly_pct
(zone-based + slightly influenced by call_count)

In [ ]:
import numpy as np

# Create base elderly % per zone
zones = df['zone_id'].unique()

zone_elderly_map = {}
for zone in zones:
    zone_elderly_map[zone] = np.random.uniform(0.05, 0.25)  # realistic range

# Map to dataset
df['elderly_pct'] = df['zone_id'].map(zone_elderly_map)

# Add slight variation based on call_count (to make it meaningful)
df['elderly_pct'] = df['elderly_pct'] + (df['call_count'] * 0.005)

# Clip values to keep realistic
df['elderly_pct'] = df['elderly_pct'].clip(0.05, 0.4)

df[['zone_id', 'call_count', 'elderly_pct']].head()

,zone_id,call_count,elderly_pct
0,Z22,1,0.231803
1,Z04,1,0.199929
2,Z06,1,0.138750
3,Z11,1,0.203478
4,Z20,1,0.191747


Create Risk Label (CORE LOGIC)  -- Target variable

In [ ]:
def assign_risk(row):
    score = 0

    # AQI impact
    if row['AQI'] > 300:
        score += 3
    elif row['AQI'] > 200:
        score += 2
    elif row['AQI'] > 100:
        score += 1

    # Temperature impact
    if row['temperature'] > 40:
        score += 2
    elif row['temperature'] > 35:
        score += 1

    # Humidity impact (extremes)
    if row['humidity'] > 80 or row['humidity'] < 20:
        score += 1

    # Elderly population impact
    if row['elderly_pct'] > 0.2:
        score += 2
    elif row['elderly_pct'] > 0.12:
        score += 1

    # Final classification
    if score >= 6:
        return "CRITICAL"
    elif score >= 4:
        return "HIGH"
    elif score >= 2:
        return "MODERATE"
    else:
        return "LOW"

# Apply function
df['risk_label'] = df.apply(assign_risk, axis=1)

df[['AQI', 'temperature', 'elderly_pct', 'risk_label']].head()

,AQI,temperature,elderly_pct,risk_label
0,383.0,19.0,0.231803,HIGH
1,383.0,19.0,0.199929,HIGH
2,383.0,19.0,0.138750,HIGH
3,383.0,19.0,0.203478,HIGH
4,383.0,19.0,0.191747,HIGH


Encode Target Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['risk_encoded'] = le.fit_transform(df['risk_label'])

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'CRITICAL': np.int64(0), 'HIGH': np.int64(1), 'LOW': np.int64(2), 'MODERATE': np.int64(3)}


Feature Selection

In [ ]:
features = [
    'AQI', 'PM2.5', 'PM10',
    'temperature', 'humidity',
    'population_density', 'elderly_pct',
    'hour'
]

X = df[features]
y = df['risk_encoded']

Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Evaluate Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

    CRITICAL       1.00      1.00      1.00      3244
        HIGH       1.00      1.00      1.00     24391
         LOW       1.00      1.00      1.00      2279
    MODERATE       1.00      1.00      1.00     21473

    accuracy                           1.00     51387
   macro avg       1.00      1.00      1.00     51387
weighted avg       1.00      1.00      1.00     51387



Test with Sample Input

In [ ]:
sample = pd.DataFrame([{
    'AQI': 320,
    'PM2.5': 180,
    'PM10': 220,
    'temperature': 38,
    'humidity': 65,
    'population_density': 12000,
    'elderly_pct': 0.22,
    'hour': 18
}])

prediction = model.predict(sample)
risk_output = le.inverse_transform(prediction)

print("Predicted Risk Level:", risk_output[0])

Predicted Risk Level: HIGH
